# Notebook 05 — Final Evaluation, Error Analysis & Demo
**Week 4**: Compare both models side-by-side, run head/tail breakdown, qualitative error review, and provide a CLI-style demo (input discharge summary → predicted ICD-10 codes).

In [1]:
# pip install -q pandas pyarrow numpy scikit-learn matplotlib seaborn transformers torch

In [2]:
# Running locally — no Drive mount needed

## 1. Config & load artifacts

In [ ]:
import numpy as np
import pandas as pd
import pickle, json, re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score
)

DATA   = '../datasets/processed'
MDL_A  = '../data/models/model_a'
MDL_B  = '../data/models/model_b'

# Labels
with open(f'{DATA}/mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
vocab  = list(mlb.classes_)
N_LABELS = len(vocab)

# Ground-truth test labels
Y_test  = np.load(f'{DATA}/Y_test.npy')

# Model A probabilities
import scipy.sparse as sp
X_test_tfidf = sp.load_npz(f'{DATA}/X_test_tfidf.npz')
with open(f'{MDL_A}/clf_sgd.pkl', 'rb') as f:
    clf_a = pickle.load(f)
P_a = clf_a.predict_proba(X_test_tfidf)
with open(f'{MDL_A}/results.json') as f:
    res_a = json.load(f)
T_a = res_a['test']['threshold']

# Model B probabilities
P_b = np.load(f'{MDL_B}/P_test.npy')
with open(f'{MDL_B}/test_results.json') as f:
    res_b = json.load(f)
T_b = res_b['threshold']

print(f'Loaded test probs: A={P_a.shape}  B={P_b.shape}  Y={Y_test.shape}')


FileNotFoundError: [Errno 2] No such file or directory: '../data/datasets/mlb.pkl'

## 2. Side-by-side metrics table

In [ ]:
def full_metrics(P, Y, t, name):
    preds = (P >= t).astype(int)
    mask  = Y.sum(0) > 0
    return {
        'Model'       : name,
        'Threshold'   : round(t, 3),
        'Micro-F1'    : round(f1_score(Y, preds, average='micro',  zero_division=0), 4),
        'Macro-F1'    : round(f1_score(Y, preds, average='macro',  zero_division=0), 4),
        'Micro-Prec'  : round(precision_score(Y, preds, average='micro', zero_division=0), 4),
        'Micro-Rec'   : round(recall_score(Y, preds, average='micro',    zero_division=0), 4),
        'Macro-AUPRC' : round(average_precision_score(Y[:, mask], P[:, mask], average='macro'), 4),
        'Micro-AUROC' : round(roc_auc_score(Y[:, mask], P[:, mask], average='micro'), 4),
    }

rows = [full_metrics(P_a, Y_test, T_a, 'Model A (TF-IDF LR)'),
        full_metrics(P_b, Y_test, T_b, 'Model B (ClinicalBERT)')]
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
results_df.to_csv(f'{BASE}/final_comparison.csv', index=False)

## 3. Top-K evaluation (both models)

In [ ]:
Y_train = np.load(f'{DATA}/Y_train.npy')
freq = Y_train.sum(0)

topk_rows = []
for K in [50, 100, 500]:
    top_idx = np.argsort(freq)[::-1][:K]
    for P, t, name in [(P_a, T_a, 'Model A'), (P_b, T_b, 'Model B')]:
        Yk = Y_test[:, top_idx]
        Pk = P[:, top_idx]
        mf1 = f1_score(Yk, (Pk >= t).astype(int), average='micro', zero_division=0)
        topk_rows.append({'K': K, 'Model': name, 'Micro-F1': round(mf1, 4)})

topk_df = pd.DataFrame(topk_rows).pivot(index='K', columns='Model', values='Micro-F1')
print(topk_df)

## 4. Head / torso / tail breakdown

In [ ]:
preds_a = (P_a >= T_a).astype(int)
preds_b = (P_b >= T_b).astype(int)
f1_a    = f1_score(Y_test, preds_a, average=None, zero_division=0)
f1_b    = f1_score(Y_test, preds_b, average=None, zero_division=0)

label_df = pd.DataFrame({'code': vocab, 'freq': freq, 'f1_a': f1_a, 'f1_b': f1_b})

print(f"{'Bucket':25s}  {'n':>6}  {'A_F1':>7}  {'B_F1':>7}")
print('-' * 55)
for lo, hi, name in [(500, 1e9, 'head (≥500)'), (100, 499, 'torso (100–499)'), (0, 99, 'tail (<100)')]:
    s = label_df[(label_df.freq >= lo) & (label_df.freq <= hi)]
    print(f'{name:25s}  {len(s):6d}  {s.f1_a.mean():7.4f}  {s.f1_b.mean():7.4f}')

# Scatter comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (vals, mname) in zip(axes, [(f1_a, 'Model A'), (f1_b, 'Model B')]):
    ax.scatter(label_df['freq'].clip(upper=2000), vals, alpha=0.25, s=6)
    ax.set_xscale('log'); ax.set_xlabel('Train freq'); ax.set_ylabel('Test F1'); ax.set_title(mname)
plt.tight_layout(); plt.savefig(f'{BASE}/head_tail_comparison.png', dpi=120); plt.show()

## 5. Qualitative error analysis (20 samples)

In [ ]:
test_clean = pd.read_parquet(f'{DATA}/cohort_test_clean.parquet')
test_clean['icd_codes'] = test_clean['icd_codes_str'].str.split('|')

import random
random.seed(42)
sample_idx = random.sample(range(len(test_clean)), 20)

for i in sample_idx[:5]:   # show first 5 for brevity
    row       = test_clean.iloc[i]
    true_set  = set(row['icd_codes']) & set(vocab)
    pred_a    = {vocab[j] for j in np.where(preds_a[i])[0]}
    pred_b    = {vocab[j] for j in np.where(preds_b[i])[0]}
    tp_a = true_set & pred_a
    fp_a = pred_a - true_set
    fn_a = true_set - pred_a
    tp_b = true_set & pred_b
    fp_b = pred_b - true_set
    fn_b = true_set - pred_b

    print(f'--- Sample {i} (hadm_id={row["hadm_id"]}) ---')
    print(f'  Note (200c): {row["clean_text"][:200]}')
    print(f'  True codes : {sorted(true_set)}')
    print(f'  Model A TP : {sorted(tp_a)}  FP: {sorted(fp_a)}  FN: {sorted(fn_a)}')
    print(f'  Model B TP : {sorted(tp_b)}  FP: {sorted(fp_b)}  FN: {sorted(fn_b)}')
    print()

## 6. Interactive demo — predict ICD-10 codes from free text

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

# ── Rebuild Model B class (same as notebook 04) ────────────────────────
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_SEQ_LEN = 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class ICDClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.bert.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask):
        cls = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.head(cls)

tokenizer_b = AutoTokenizer.from_pretrained(MODEL_NAME)
model_b = ICDClassifier(MODEL_NAME, len(vocab)).to(device)
model_b.load_state_dict(torch.load(f'{MDL_B}/best_model.pt', map_location=device))
model_b.eval()

# TF-IDF vectorizer + LR model A
with open(f'{DATA}/tfidf_vectorizer.pkl', 'rb') as f:
    vec_a = pickle.load(f)
print('Demo models loaded.')

In [ ]:
def clean_text(text):
    text = re.sub(r'\[\*\*[^\]]*\*\*\]', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s.,;:\-/]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def predict_icd(discharge_text: str, top_n: int = 10):
    """
    Returns top-N predicted ICD-10 codes with probabilities from both models.
    """
    cleaned = clean_text(discharge_text)

    # Model A
    xv = vec_a.transform([cleaned])
    pa = clf_a.predict_proba(xv)[0]

    # Model B
    enc = tokenizer_b(cleaned, max_length=MAX_SEQ_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = model_b(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    pb = torch.sigmoid(logits).cpu().numpy()[0]

    # Rank by Model B score (use as primary)
    ranked = sorted(zip(vocab, pa, pb), key=lambda x: x[2], reverse=True)[:top_n]

    print(f"{'ICD-10 Code':15s}  {'Model A (LR)':>13}  {'Model B (BERT)':>14}")
    print('-' * 48)
    for code, sa, sb in ranked:
        print(f'{code:15s}  {sa:13.4f}  {sb:14.4f}')

# ── Example prediction ────────────────────────────────────────────
SAMPLE_NOTE = """
Patient is a 72-year-old male with a history of type 2 diabetes mellitus, chronic kidney disease
stage 3, and hypertension who presented with shortness of breath and bilateral lower extremity
edema. Hospital course was notable for decompensated heart failure treated with IV furosemide.
Discharge diagnoses: acute on chronic systolic heart failure, type 2 diabetes mellitus,
hypertensive heart disease, CKD stage 3.
"""

predict_icd(SAMPLE_NOTE, top_n=10)